# Lab 8 — Where, How Many, Which Pixels

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/INCORTX/INCORTX.github.io/blob/master/ComputerVision/session-08/lab.ipynb)

**Computer Vision · Session 8 — Detection & Segmentation**

Click the badge to open it in Colab, or download it from https://classes.incortx.com/ComputerVision/session-08/lab.ipynb and run it in
Jupyter — it needs only `numpy`, `opencv-python` and `matplotlib`, which Colab already has.

Session 8 measured a detector rather than training one. This notebook rebuilds that measurement
from scratch: the same composed scene, the same sliding-window detector, and then every metric of the
session — **IoU, NMS, precision, recall, AP, mIoU** — written by you in a few lines each and checked
against a library or against a number you can work out by hand.

| Part | Slides | What you write |
|---|---|---|
| **0** | 4, 14 | the scene and the detector (2,985 raw boxes) |
| **A** | 11–13 | `iou()` · how forgiving it is · `match()` at three thresholds |
| **B** | 21–23 | `nms()` checked against OpenCV · breaking it · the two knobs |
| **C** | 32–34 | `ap()` from a ranked list · order vs count · AP at four IoUs |
| **D** | 43–45 | mIoU on masks · why accuracy lies · the merge mIoU cannot see |
| **2** | 55–58 | the evaluation you would hand to a client, over 20 frames |

**Runtime:** under a minute on a laptop CPU. **Downloads:** two OpenCV sample images (≈ 0.4 MB). No GPU, no weights.

### How to use this notebook

Run the cells top to bottom. Nothing is left blank: every cell runs as written, and the numbers we
measured are written underneath each one — **predict before you run, then check yourself.** The
"Your Turn" cells carry the questions the slides ask; the code is complete, and the question is what
the output means.

**The numbers in this notebook are real.** Everything quoted here came from running this file. The
scene is composed deterministically (seeded noise), so you should get the same numbers to the last
digit; if one moves, an OpenCV version changed how it resizes, and it is worth finding out which.

---
# Part 0 — The scene, and the detector that fires 2,985 times

The deck's test scene is **composed**: six real copies of an OpenCV sample object are pasted onto a
real photograph at known positions and scales, plus three copies of a second object ("panel") and
some noise. That is done for one reason — **the ground truth is exact**, so every precision and recall
below means what it says instead of depending on somebody's hand-drawn boxes.

In [ ]:
import os, urllib.request
import numpy as np, cv2
import matplotlib.pyplot as plt

OPENCV = "https://raw.githubusercontent.com/opencv/opencv/4.x/samples/data/{}"
for name in ("box.png", "building.jpg"):
    if not os.path.exists(name):
        urllib.request.urlretrieve(OPENCV.format(name), name)
obj = cv2.imread("box.png", cv2.IMREAD_GRAYSCALE)        # the carton
bg  = cv2.imread("building.jpg", cv2.IMREAD_GRAYSCALE)   # the background
print("opencv", cv2.__version__, "· object", obj.shape, "· background", bg.shape)

In [ ]:
BASE = (86, 62)                       # the template size, in pixels (w, h)
PLACED = [(40, 40, 0.80, 1.00), (250, 55, 0.90, 0.90), (430, 130, 1.00, 1.05),
          (120, 250, 1.12, 0.95), (395, 300, 1.25, 0.85), (215, 330, 0.95, 1.10)]
PANELS = [(300, 175), (60, 150), (500, 240)]              # a second class, 66x48

def compose(placed=PLACED, seed=7):
    """Paste cartons (x, y, scale, brightness) and panels onto the photo. Returns (image, gt boxes)."""
    rng = np.random.default_rng(seed)
    im = cv2.resize(bg, (640, 440), interpolation=cv2.INTER_AREA)
    im = cv2.GaussianBlur(im.astype(np.float32), (0, 0), 1.2) * 0.75 + 40
    gt = []
    for (x, y, s, br) in placed:
        w, h = int(BASE[0] * s), int(BASE[1] * s)
        im[y:y + h, x:x + w] = cv2.resize(obj, (w, h), interpolation=cv2.INTER_AREA).astype(np.float32) * br
        gt.append((x, y, w, h))
    panel = cv2.resize(bg[120:200, 300:410], (66, 48), interpolation=cv2.INTER_AREA).astype(np.float32)
    for (x, y) in PANELS:
        im[y:y + 48, x:x + 66] = panel
    im = np.clip(im + rng.normal(0, 4, im.shape), 0, 255).astype(np.uint8)
    return im, gt

scene, GT = compose()
template = cv2.resize(obj, BASE, interpolation=cv2.INTER_AREA)

def detect(im, score_thr=0.35):
    """A real sliding-window detector: the template scored at every position."""
    res = cv2.matchTemplate(im, template, cv2.TM_CCOEFF_NORMED)
    ys, xs = np.where(res > score_thr)
    boxes = [(int(x), int(y), BASE[0], BASE[1]) for x, y in zip(xs, ys)]
    return boxes, res[ys, xs].astype(float)

raw_boxes, raw_scores = detect(scene, 0.35)
print(len(raw_boxes), "raw boxes above score 0.35 ·", len(detect(scene, 0.45)[0]), "above 0.45")
print("ground truth:", GT)

In [ ]:
vis = cv2.cvtColor(scene, cv2.COLOR_GRAY2RGB)
for (x, y, w, h) in GT:
    cv2.rectangle(vis, (x, y), (x + w, y + h), (40, 200, 40), 2)
plt.figure(figsize=(8, 5.5)); plt.imshow(vis); plt.axis("off")
plt.title("the composed scene · six cartons, six exact ground-truth boxes"); plt.show()

**2,985 boxes above 0.35, 157 above 0.45** — the two numbers the deck's hooks quote (slides 4 and 14
differ only in that score cut). Six objects are there. Everything from here on is about turning that
pile into a count, and then scoring the count honestly.

The list below is what the session's detector keeps after the tidy-up you will write in Part B — the
16 boxes every table in the deck is built from. It is given here so Part A can use it before Part B
exists; Part B checks that your own NMS reproduces it exactly.

In [ ]:
# the session's 16 detections after NMS (x, y at the 86x62 template size), ranked by score
XY = [(430,130), (128,250), (213,328), (395,300), (524,273), (241,55), (511,302), (529,239),
      (402,257), (511,200), (67,187), (498,4), (526,330), (33,40), (420,191), (68,114)]
S  = [.997, .545, .537, .476, .465, .453, .425, .405, .397, .382, .381, .380, .374, .370, .362, .353]
DETS = [((x, y, 86, 62), s) for (x, y), s in zip(XY, S)]
print(len(DETS), "detections · top score", DETS[0][1], "· lowest", DETS[-1][1])

---
# Part A — IoU: one number for "is this box right?"

### Run — IoU in four lines (slide 11)

In [ ]:
def iou(a, b):                        # each box is (x, y, w, h)
    ix = max(0, min(a[0]+a[2], b[0]+b[2]) - max(a[0], b[0]))
    iy = max(0, min(a[1]+a[3], b[1]+b[3]) - max(a[1], b[1]))
    inter = ix * iy
    return inter / (a[2]*a[3] + b[2]*b[3] - inter)

print(iou((0,0,10,10), (0,0,10,10)))      # identical
print(iou((0,0,10,10), (20,20,10,10)))    # no overlap
print(iou((0,0,10,10), (5,0,10,10)))      # half overlap
print(iou((0,0,10,10), (-5,-5,20,20)))    # one inside the other

**1.0 · 0.0 · 0.333… · 0.25.** Check the third by hand: overlap 5×10 = 50, union 100 + 100 − 50 = 150.
The fourth is the one to remember — a box that contains the object perfectly but is four times too big
scores 0.25, not 1.0.

### Explore — how forgiving is IoU, really? (slide 12)

In [ ]:
gt = (100, 100, 80, 60)
print("shifted:", [round(iou((100+d, 100, 80, 60), gt), 2) for d in (4, 8, 16, 32)])
print("scaled: ", [round(iou((100, 100, int(80*s), int(60*s)), gt), 2) for s in (1.1, 1.25, 1.5, 2.0)])

Shifting by 4, 8, 16, 32 pixels: **0.90 · 0.82 · 0.67 · 0.43**. Scaling by 1.1× to 2×: **0.83 · 0.64 · 0.44 · 0.25**.
IoU falls fast — a box 25% too big is already below the usual 0.5 bar — and getting the *size* wrong
hurts more than getting the position wrong, which is why detectors predict box shape so carefully and
why anchors exist.

### Your Turn — the threshold you would defend (slide 13)

**Predict first:** for counting cartons on a pallet, is IoU 0.5 too strict, about right, or too loose?

`match()` is the standard bookkeeping: walk the predictions in score order, give each one its best
*still unclaimed* ground-truth box, and call it a TP if that IoU clears the threshold. The
`one_to_one` switch lets you remove the "one prediction, one object" rule and see what breaks.

In [ ]:
def match(dets, gt, thr=0.5, one_to_one=True):
    """Greedy TP/FP assignment. Returns (hit flags in score order, number of objects missed)."""
    used, hits = set(), []
    for box, _ in dets:
        cands = [(iou(box, g), i) for i, g in enumerate(gt) if not (one_to_one and i in used)]
        best, bi = max(cands) if cands else (0.0, -1)
        hits.append(best >= thr)
        if best >= thr:
            used.add(bi)
    return hits, len(gt) - len(used)

for thr in (0.3, 0.5, 0.7):
    hits, missed = match(DETS, GT, thr)
    print(f"IoU >= {thr}: {sum(hits)} TP · {len(hits) - sum(hits)} FP · {missed} missed")

raw = sorted(zip(raw_boxes, raw_scores), key=lambda d: -d[1])   # the 2,985 boxes, no NMS
for rule in (True, False):
    hits, _ = match(raw, GT, 0.5, one_to_one=rule)
    print(f"raw pile, one-match rule {'on ' if rule else 'off'}: {sum(hits):4d} TP on {len(GT)} objects")

**6 TP / 10 FP / 0 missed** at 0.3 and at 0.5; **4 TP / 12 FP / 2 missed** at 0.7 — the three panels of
slide 8, from your own code. On the raw pile the one-match rule is what keeps the count honest: with it,
2,985 boxes still yield **6 TP**; without it, **656 of them count as "correct"** on six objects, and a
detector could score perfectly just by predicting duplicates. That rule is what makes precision mean
something.

And the threshold you would defend: **counting tolerates 0.3–0.5** because you only need the count; a
robot arm that must grasp the carton needs **0.75 or more**, because a box that is 30% off is a
collision.

---
# Part B — One object, many boxes: Non-Maximum Suppression

### Run — write NMS, then check it against OpenCV (slide 21)

In [ ]:
def nms(boxes, scores, thr=0.3):
    order = np.argsort(scores)[::-1]                  # highest score first
    keep = []
    while len(order):
        i = order[0]; keep.append(int(i))
        order = np.array([j for j in order[1:] if iou(boxes[i], boxes[j]) <= thr], int)
    return keep

boxes  = [[10,10,50,50], [14,12,50,50], [200,20,50,50], [18,8,50,50]]
scores = [0.90, 0.85, 0.75, 0.60]
print("mine  ", sorted(nms(boxes, scores)))
print("opencv", sorted(int(i) for i in np.array(cv2.dnn.NMSBoxes(boxes, scores, 0.3, 0.3)).ravel()))

# now on the real pile — and check it is the list the deck used
keep = nms(raw_boxes, raw_scores, 0.3)
mine = sorted(((raw_boxes[i], raw_scores[i]) for i in keep), key=lambda d: -d[1])
print(len(raw_boxes), "raw boxes ->", len(keep), "after NMS")
assert [b for b, _ in mine] == [b for b, _ in DETS], "your NMS does not reproduce the session's list"
print("matches the session's 16 detections exactly")

Both print **[0, 2]**: boxes 1 and 3 overlap box 0 heavily and are suppressed, box 2 is elsewhere and
survives. On the real pile the same nine lines turn **2,985 boxes into 16** — the very list Part A used.

### Explore — break NMS on purpose (slide 22)

In [ ]:
# Two people standing shoulder to shoulder: their boxes really do overlap.
two = [[100,100,60,140], [122,100,60,140]]
sc  = [0.9, 0.8]
print("their IoU:", round(iou(two[0], two[1]), 2))
for thr in (0.2, 0.3, 0.5, 0.7):
    print(f"thr {thr}: {len(nms(two, sc, thr))} box(es) kept")

Their IoU is **0.46**, so at the popular default of 0.3 **NMS deletes one of the two people** — and no
amount of retraining fixes it, because the model was right and the post-processing threw the answer
away. Crowded scenes are where detectors quietly under-count.

### Your Turn — the two knobs, and what each one costs (slide 23)

**Predict first:** raising the *score* threshold and raising the *NMS* threshold both reduce clutter.
Do they reduce the same clutter?

In [ ]:
# knob 1 — the score threshold, NMS fixed at 0.3 (filtering the final list is the same thing)
for t in (0.35, 0.40, 0.45, 0.50, 0.55, 0.60):
    keep = [d for d in DETS if d[1] >= t]
    hits, _ = match(keep, GT, 0.5)
    print(f"score >= {t:.2f}: {len(keep):2d} boxes, {sum(hits)} correct")
print()
# knob 2 — the NMS threshold, score fixed at 0.35 (this one needs the raw pile)
for t in (0.1, 0.2, 0.3, 0.5, 0.7):
    keep = nms(raw_boxes, raw_scores, t)
    dets = sorted(((raw_boxes[i], raw_scores[i]) for i in keep), key=lambda d: -d[1])
    hits, _ = match(dets, GT, 0.5)
    print(f"NMS thr {t:.1f}: {len(keep):3d} boxes, {sum(hits)} correct")

Knob 1 (score): **16 → 8 → 6 → 3 → 1 → 1** boxes kept, and the correct ones go **6 → 5 → 5 → 3 → 1 → 1**.
One step, 0.35 → 0.40, deletes eight boxes: seven false alarms and one real carton (the faint one at
score 0.37). Knob 2 (NMS): **14 · 15 · 16 · 22 · 45** boxes, and the correct count is **6 every time** —
raising the NMS threshold never removes a single false alarm, it only lets more duplicates through.

So: **the score threshold removes low-confidence false alarms; the NMS threshold removes overlapping
duplicates.** They are not interchangeable. A shelf-counting app that reports 16 cartons when there are
6 should raise the *score* threshold — and the cost is that a genuinely faint carton at the back of the
shelf disappears with the false alarms, which is exactly the trade Part C measures.

---
# Part C — Scoring a detector: precision, recall and AP

### Run — precision, recall and AP from a ranked list (slide 32)

`ap()` is the whole of the standard definition: cumulative TP and FP down the ranked list, recall and
precision at every step, precision made monotonic *from the right*, then the area under the stairs.

In [ ]:
def ap(hits, n_objects):
    """Average precision (all-point interpolation) from hit flags in score order."""
    h = np.array(hits, int)
    tp, fp = np.cumsum(h), np.cumsum(1 - h)
    recall, precision = tp / n_objects, tp / (tp + fp)
    mrec = np.r_[0, recall, 1]; mpre = np.r_[0, precision, 0]
    for i in range(len(mpre) - 2, -1, -1):
        mpre[i] = max(mpre[i], mpre[i+1])            # make precision monotonic
    i = np.where(mrec[1:] != mrec[:-1])[0]
    return float(((mrec[i+1] - mrec[i]) * mpre[i+1]).sum()), recall, precision

# the session's 16 detections, already ranked: 1 = matched a real carton at IoU 0.5
hits = [1,1,1,1,0,1,0,0,0,0,0,0,0,1,0,0]
a, recall, precision = ap(hits, 6)
print("recall   ", np.round(recall, 2))
print("precision", np.round(precision, 2))
print("AP =", round(a, 3))

**AP = 0.877** — the same number as the figure on slide 27. Note the monotonic step: the standard
definition looks *forward* along the curve and uses the best precision still achievable, which is why
the curve is drawn as stairs and not as a wobble.

### Explore — does the order matter more than the count? (slide 33)

In [ ]:
# Same 6 hits and 10 misses every time — only the ranking changes.
for name, h in [
    ("hits first",  [1]*6 + [0]*10),
    ("as measured", [1,1,1,1,0,1,0,0,0,0,0,0,0,1,0,0]),
    ("alternating", [1,0]*6 + [0]*4),
    ("hits last",   [0]*10 + [1]*6)]:
    print(f"{name:12s} AP = {ap(h, 6)[0]:.3f}")

**1.000 · 0.877 · 0.657 · 0.375.** Every row has the same six correct detections and the same ten
mistakes — only the confidence ranking differs, and AP moves from 1.00 to 0.375. AP is not measuring
"how often is it right" but **"does it know when it is right"**.

### Your Turn — report a detector the way a reviewer would want it (slide 34)

**Predict first:** your model reports mAP 0.72 and a competitor reports 0.78. Is theirs better?

In [ ]:
def ap_at(dets, gt, thr):
    hits, _ = match(dets, gt, thr)
    return ap(hits, len(gt))[0]

aps = [ap_at(DETS, GT, t) for t in (0.3, 0.5, 0.75, 0.9)]
print("AP at IoU 0.3 / 0.5 / 0.75 / 0.9:", np.round(aps, 3))
print("their mean (a small COCO-style AP@[.5:.95]):", round(float(np.mean(aps)), 3))
print("drop the two lowest-scoring detections, AP@0.5:", round(ap_at(DETS[:-2], GT, 0.5), 3))
print("drop the two HIGHEST-scoring instead,   AP@0.5:", round(ap_at(DETS[2:], GT, 0.5), 3))

**0.877 · 0.877 · 0.611 · 0.167** — AP survives the move from 0.3 to 0.5 untouched, then collapses.
Their mean is **0.633**, closest to the single threshold 0.75: averaging over thresholds is much harsher
than the usual 0.5, which is the point of it.

Dropping the two lowest-scoring detections changes AP@0.5 by **exactly nothing** (both were false
positives at the bottom of the list, where precision had nothing left to lose) — low-confidence mistakes
are nearly free, which is why detectors output so many of them. Drop the two *highest*-scoring instead
and AP falls to **0.514**, because those were two of the six cartons.

The one-line results statement: *"AP = 0.877 at IoU 0.5 on 6 objects; 0.611 at IoU 0.7."* And the answer
to the hook: **you cannot tell** whether 0.78 beats 0.72 without their IoU threshold, dataset and class list.

---
# Part D — Down to the pixel: mIoU

### Run — mIoU on two masks (slide 43)

In [ ]:
gt   = np.zeros((10, 10), int); gt[2:8, 2:6] = 1      # 24 object pixels
pred = np.zeros((10, 10), int); pred[3:8, 3:8] = 1    # 25, shifted

for c in (0, 1):
    inter = ((gt == c) & (pred == c)).sum()
    union = ((gt == c) | (pred == c)).sum()
    print(f"class {c}: IoU = {inter/union:.3f}")

ious = [((gt==c)&(pred==c)).sum() / ((gt==c)|(pred==c)).sum() for c in (0,1)]
print("mIoU    =", round(float(np.mean(ious)), 3))
print("accuracy=", round(float((gt == pred).mean()), 3))

**class 0: 0.776 · class 1: 0.441 · mIoU 0.609 · accuracy 0.810.** Two overlapping rectangles, and the
object IoU is already below 0.5. Accuracy says 81%; the class you care about says 0.44.

### Explore — make the background bigger and watch accuracy lie (slide 44)

In [ ]:
for n in (10, 20, 50, 200):
    gt   = np.zeros((n, n), int); gt[2:8, 2:6] = 1
    pred = np.zeros((n, n), int); pred[3:8, 3:8] = 1
    ious = [((gt==c)&(pred==c)).sum() / ((gt==c)|(pred==c)).sum() for c in (0,1)]
    print(f"{n:4d}x{n}: accuracy {float((gt==pred).mean()):.3f}  "
          f"object IoU {ious[1]:.3f}  mIoU {float(np.mean(ious)):.3f}")

The object and its error never change; only the empty space around them grows. Accuracy climbs
**0.810 → 0.953 → 0.992 → 1.000** while the object IoU stays pinned at **0.441**. This is the normal
situation in medical imaging and defect inspection, where the thing you are looking for is a tiny
fraction of the frame.

### Your Turn — pick the task, then defend the metric (slide 45)

**Predict first:** for "what percentage of this crop field is diseased", is instance segmentation worth
the extra labelling?

Two cartons stand touching, one pixel apart. The segmenter paints them as one blob — every pixel right
except the seam. Does the standard metric notice?

In [ ]:
gt = np.zeros((12, 12), int); gt[3:9, 2:6] = 1; gt[3:9, 7:11] = 1     # two objects, a 1-px seam
pred = np.zeros((12, 12), int); pred[3:9, 2:11] = 1                  # one merged blob

ious = [((gt==c)&(pred==c)).sum() / ((gt==c)|(pred==c)).sum() for c in (0, 1)]
print(f"mIoU {float(np.mean(ious)):.3f} · accuracy {float((gt==pred).mean()):.3f}")
n_true, _ = cv2.connectedComponents(gt.astype(np.uint8))
n_pred, _ = cv2.connectedComponents(pred.astype(np.uint8))
print(f"objects counted: truth {n_true - 1} · prediction {n_pred - 1}")

**mIoU 0.913, accuracy 0.958** — both still look fine, because almost every pixel carries the right
class and the only error is a six-pixel seam. The component count catches it at once: **2 versus 1.**
Choose a metric that can see the failure you care about, and choose the task before the metric.

For the record: (a) count cars at a junction → detection + mAP · (b) % of a field diseased →
semantic + mIoU · (c) each tumour's area → instance + per-object IoU · (d) is there a helmet →
classification + accuracy. And with a budget to label 500 images once: **boxes**, unless the question is
about area or shape — boxes cover far more variation for the same effort, and what you lose is the
ability to answer any question about area.

---
# Part 2 — The evaluation you would hand over (slide 58)

The case: one fixed camera over a supermarket shelf, one frame a minute, products standing touching
each other. Part A of the case (slides 56–57) settled the decisions — detection, evaluate at IoU 0.5,
run NMS high (0.5) because neighbours genuinely overlap, favour recall. Part B is the evaluation
itself, over twenty frames.

There is no real shelf camera in this lab, so the twenty frames are twenty composed scenes: 4–7
cartons each at random positions and scales, exact ground truth every time.

In [ ]:
def random_scene(seed, n_obj):
    """A composed frame with n_obj non-overlapping cartons at random positions and scales."""
    rng, placed = np.random.default_rng(seed), []
    taken = [(x, y, 66, 48) for x, y in PANELS]              # the panels are pasted last — keep clear
    while len(placed) < n_obj:
        s = float(rng.uniform(0.8, 1.25)); w, h = int(86 * s), int(62 * s)
        x, y = int(rng.integers(0, 640 - w)), int(rng.integers(0, 440 - h))
        if all(iou((x, y, w, h), t) == 0 for t in taken):
            taken.append((x, y, w, h)); placed.append((x, y, s, float(rng.uniform(0.85, 1.1))))
    return compose(placed, seed)

frames = [random_scene(k, 4 + k % 4) for k in range(20)]     # 20 frames, 4-7 cartons each
print(sum(len(gt) for _, gt in frames), "cartons across 20 frames")

In [ ]:
def evaluate(frames, nms_thr=0.5, iou_thr=0.5):
    """NMS per frame, match per frame, then ONE ranked list across all frames -> AP."""
    ranked, n_gt, per_frame = [], 0, []
    for im, gt in frames:
        b, s = detect(im, 0.35)
        keep = nms(b, s, nms_thr)
        dets = sorted(((b[i], s[i]) for i in keep), key=lambda d: -d[1])
        hits, _ = match(dets, gt, iou_thr)
        ranked += [(sc, h) for (_, sc), h in zip(dets, hits)]
        per_frame.append(ap(hits, len(gt))[0])
        n_gt += len(gt)
    ranked.sort(key=lambda t: -t[0])                          # global ranking
    a, rec, prec = ap([h for _, h in ranked], n_gt)
    return a, rec, prec, ranked, n_gt, float(np.mean(per_frame))

AP5, rec, prec, ranked, n_gt, per_frame_mean = evaluate(frames)
print(f"{n_gt} cartons in 20 frames · AP@0.5 = {AP5:.3f}   (per-frame AP averaged: {per_frame_mean:.3f})")
for t in (0.3, 0.75):
    print(f"AP@{t} = {evaluate(frames, iou_thr=t)[0]:.3f}")

# the product decision: the cheapest threshold that still finds 95% (or 90%) of the cartons
for target in (0.95, 0.90):
    if (rec >= target).any():
        i = int(np.argmax(rec >= target))
        print(f"recall >= {target:.2f}: score >= {ranked[i][0]:.2f} gets recall {rec[i]:.2f} at precision {prec[i]:.2f}")
    else:
        print(f"recall >= {target:.2f}: never reached — the curve tops out at {rec.max():.3f} "
              f"({int(round(rec.max() * n_gt))} of {n_gt} cartons)")

In [ ]:
def envelope(r, p):
    """The interpolated curve AP integrates: precision made monotonic from the right."""
    mrec, mpre = np.r_[0, r, 1], np.r_[0, p, 0]
    for i in range(len(mpre) - 2, -1, -1):
        mpre[i] = max(mpre[i], mpre[i+1])
    end = int(np.argmax(mrec[:-1] >= mrec[:-1].max())) + 1    # trailing FPs add no recall
    return mrec[:end], mpre[:end]

plt.figure(figsize=(6, 4))
for t in (0.3, 0.5, 0.75):
    a, r, p, *_ = evaluate(frames, iou_thr=t)
    plt.step(*envelope(r, p), where="pre", lw=2, label=f"IoU {t} · AP = {a:.3f}")
plt.xlabel("recall"); plt.ylabel("precision"); plt.ylim(0, 1.05); plt.legend(); plt.grid(alpha=.3)
plt.title("the same detector on 20 shelf frames, scored three ways"); plt.show()

**110 cartons over 20 frames · AP@0.5 = 0.878 · AP@0.3 = 0.878 · AP@0.75 = 0.566.** Averaging the
per-frame APs instead gives **0.884** — the flattering number: ranking inside each frame means a
confident mistake in one frame never has to compete with a hesitant correct answer in another.

The deliverable is not the AP but the operating point, and here the requirement fails: **no score
threshold reaches 95% recall** — the curve tops out at 0.945, because 6 of the 110 cartons are never
found at any score (most of them stand touching a neighbour, and NMS at 0.5 keeps one box for the pair).
The nearest honest offer is **recall 0.90 at score ≥ 0.39, where only 47% of what it reports is real**.
That pair, and what the missed 10% and the extra 53% cost per shift, is the product decision — and "the
system cannot meet the 95% requirement as specified" is a result, not a failure of the evaluation.

The one-line statement for the report: *"AP = 0.878 at IoU 0.5 on 110 objects in 20 frames; 0.566 at
IoU 0.75; recall tops out at 0.945; at 90% recall, precision 0.47."* And the honest caveat: **all twenty
frames are composed from one object on one background** — a real shelf will have glare, occlusion and
products that are not this carton, and every number above is an upper bound until it is measured there.

---
*Images: OpenCV samples (Apache-2.0). Every number in this notebook was produced by the cells above.*